# Sesión 3: Modelo de Datos y Estructura
## Clase 3 - Fundamentos de Bases de Datos

En esta sesión aprenderemos:
- Elementos de un modelo de datos (entidades, atributos, identificadores)
- Tipos de relaciones entre entidades (1:1, 1:N, N:M)
- Diagramas Entidad-Relación (DER)
- Diseño de base de datos académica

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("✅ Conexión SQLite establecida")

## 1. MODELO DE DATOS - CONCEPTOS FUNDAMENTALES

### ¿Qué es un modelo de datos?

Un modelo de datos es una **representación lógica y estructurada** de la información relevante para un sistema. Organiza y documenta cómo se almacenan, agrupan y relacionan los datos.

### Tipos de Modelos (Slide 6)

1. **Modelo Conceptual:** Visión general sin detalles técnicos
2. **Modelo Lógico:** Entidades, atributos, relaciones detalladas
3. **Modelo Físico:** Implementación en SGBD específico

### Componentes Esenciales (Slide 8)

| Elemento | Descripción | Ejemplo |
|----------|-------------|----------|
| **Entidad** | Objeto real o concepto | Estudiante |
| **Atributo** | Propiedad de una entidad | nombre, correo |
| **Identificador** | Atributo único | id_estudiante |
| **Relación** | Vínculo entre entidades | se matricula en |
| **Cardinalidad** | Cuántas instancias se relacionan | 1:1, 1:N, N:M |

## 2. ENTIDADES, ATRIBUTOS E IDENTIFICADORES (Slides 14-23)

### ¿Qué es una entidad? (Slide 15)

Representación de un objeto real del que necesitamos almacenar información.

**Ejemplos:** Persona, Producto, Curso, Cliente, Vehículo

### Criterios para identificar una entidad (Slide 16)

✓ Tiene existencia propia en el dominio  
✓ Es relevante para las funciones del negocio  
✓ Almacena información persistente (no transitoria)  
✓ Participa activamente en procesos del negocio  
✓ Puede relacionarse con otras entidades

In [ ]:
# Ejemplo: Crear entidad ESTUDIANTE (Slide 11, 23)
cursor.execute('''
    CREATE TABLE estudiante (
        id_estudiante INT PRIMARY KEY,
        nombre VARCHAR(100),
        correo VARCHAR(100),
        fecha_nacimiento DATE,
        carrera VARCHAR(100)
    )
''')

print("✅ Entidad ESTUDIANTE creada")
print("\nAtributos:")
print("  - id_estudiante (IDENTIFICADOR)")
print("  - nombre")
print("  - correo")
print("  - fecha_nacimiento")
print("  - carrera")

### Tipos de Atributos (Slide 19)

- **Simples:** No se pueden descomponer (edad)
- **Compuestos:** Se pueden dividir (nombre completo)
- **Derivados:** Se obtienen de otros datos (edad desde fecha_nacimiento)
- **Multivaluados:** Múltiples valores (varios correos)

### Identificadores - Clave Primaria (Slide 21-23)

**Características clave:**
- Debe ser **único**
- No debe cambiar con el tiempo
- No debe ser nulo

**Tipos:**
- **Natural:** Información existente (RUT, número de serie)
- **Artificial:** Generado por el sistema (id autoincremental)

## 3. TIPOS DE RELACIONES (Slides 26-30)

### Relación 1:1 (Uno a Uno)
Cada instancia de una entidad se relaciona con UNA sola instancia de otra.

**Ejemplo:** Empleado ↔ Contrato

In [ ]:
# Ejemplo: Relación 1:1
cursor.execute('''
    CREATE TABLE contrato (
        id_contrato INT PRIMARY KEY,
        id_estudiante INT UNIQUE,
        numero_contrato VARCHAR(50),
        fecha_inicio DATE,
        FOREIGN KEY (id_estudiante) REFERENCES estudiante(id_estudiante)
    )
''')

print("✅ Relación 1:1 (Estudiante - Contrato)")
print("\nCaracterística: UNIQUE en FK asegura 1:1")

### Relación 1:N (Uno a Muchos)
Una instancia de una entidad se relaciona con MÚLTIPLES instancias de otra.

**Ejemplo:** Docente → Cursos

**MÁS COMÚN en bases de datos**

In [ ]:
# Ejemplo: Relación 1:N
cursor.execute('''
    CREATE TABLE curso (
        id_curso INT PRIMARY KEY,
        nombre VARCHAR(100),
        horas INT,
        id_docente INT,
        FOREIGN KEY (id_docente) REFERENCES docente(id_docente)
    )
''')

# Primero crear tabla docente
cursor.execute('''
    CREATE TABLE docente (
        id_docente INT PRIMARY KEY,
        nombre VARCHAR(100),
        especialidad VARCHAR(100)
    )
''')

print("✅ Relación 1:N (Docente - Cursos)")
print("\nCaracterística: FK en tabla 'muchos' (curso)")

### Relación N:M (Muchos a Muchos)
Múltiples instancias de una entidad se relacionan con múltiples de otra.

**Ejemplo:** Estudiantes ↔ Cursos

**Requiere tabla intermedia (tabla puente)**

In [ ]:
# Ejemplo: Relación N:M (Slide 27-28)
cursor.execute('''
    CREATE TABLE matricula (
        id_matricula INT PRIMARY KEY,
        id_estudiante INT,
        id_curso INT,
        fecha_inscripcion DATE,
        FOREIGN KEY (id_estudiante) REFERENCES estudiante(id_estudiante),
        FOREIGN KEY (id_curso) REFERENCES curso(id_curso)
    )
''')

print("✅ Relación N:M (Estudiante - Curso vía MATRÍCULA)")
print("\nCaracterística: Tabla intermedia con 2 FK")
print("\nUn estudiante → Muchos cursos")
print("Un curso → Muchos estudiantes")

## 4. CÓDIGO SQL REAL (Slide 28)

Implementación completa de la relación Estudiante-Curso

In [ ]:
# Insertar datos de ejemplo
estudiantes = [
    (1, 'Ana García', 'ana@email.com', '2000-05-15', 'Informática'),
    (2, 'Carlos López', 'carlos@email.com', '2001-08-22', 'Informática'),
    (3, 'María Rodríguez', 'maria@email.com', '2000-03-10', 'Comercial')
]

docentes = [
    (1, 'Dr. García', 'Programación'),
    (2, 'Dra. López', 'Bases de Datos'),
    (3, 'Prof. Martínez', 'Comercio Electrónico')
]

cursos_data = [
    (1, 'SQL Básico', 60, 2),
    (2, 'Python', 80, 1),
    (3, 'E-commerce', 40, 3)
]

matriculas = [
    (1, 1, 1, '2024-01-10'),
    (2, 1, 2, '2024-01-10'),
    (3, 2, 1, '2024-01-15'),
    (4, 2, 3, '2024-01-20'),
    (5, 3, 3, '2024-01-18')
]

cursor.executemany('INSERT INTO estudiante VALUES (?, ?, ?, ?, ?)', estudiantes)
cursor.executemany('INSERT INTO docente VALUES (?, ?, ?)', docentes)
cursor.executemany('INSERT INTO curso VALUES (?, ?, ?, ?)', cursos_data)
cursor.executemany('INSERT INTO matricula VALUES (?, ?, ?, ?)', matriculas)
conn.commit()

print("✅ Datos de ejemplo insertados")
print(f"   - {len(estudiantes)} estudiantes")
print(f"   - {len(docentes)} docentes")
print(f"   - {len(cursos_data)} cursos")
print(f"   - {len(matriculas)} matrículas")

## 5. VALIDAR LAS RELACIONES

Consultar la relación N:M creada

In [ ]:
# Ver relación N:M (Estudiantes inscritos en Cursos)
df_matriculas = pd.read_sql_query(
    '''SELECT 
        e.nombre as estudiante,
        c.nombre as curso,
        d.nombre as docente,
        m.fecha_inscripcion
    FROM matricula m
    JOIN estudiante e ON m.id_estudiante = e.id_estudiante
    JOIN curso c ON m.id_curso = c.id_curso
    JOIN docente d ON c.id_docente = d.id_docente
    ORDER BY e.nombre, c.nombre''',
    conn
)

print("Relación N:M - Estudiantes inscritos en Cursos:")
print(df_matriculas.to_string(index=False))

## 6. ANÁLISIS DE ENTIDADES Y RELACIONES

Resumen del modelo académico diseñado

In [ ]:
print("MODELO DE DATOS ACADÉMICO")
print("="*70)

print("\n1. ENTIDADES IDENTIFICADAS:")
print("   - ESTUDIANTE (entidad principal)")
print("   - DOCENTE (entidad principal)")
print("   - CURSO (entidad principal)")
print("   - MATRÍCULA (entidad relacional N:M)")
print("   - CONTRATO (entidad relacional 1:1)")

print("\n2. RELACIONES:")
print("   - Estudiante ← 1:1 → Contrato")
print("   - Docente ← 1:N → Curso")
print("   - Estudiante ← N:M → Curso (vía MATRÍCULA)")

print("\n3. ESTADÍSTICAS:")
stats = pd.read_sql_query(
    '''SELECT 
        'Estudiantes' as tipo, COUNT(*) as cantidad FROM estudiante
    UNION ALL
    SELECT 'Docentes', COUNT(*) FROM docente
    UNION ALL
    SELECT 'Cursos', COUNT(*) FROM curso
    UNION ALL
    SELECT 'Matrículas', COUNT(*) FROM matricula''',
    conn
)
print(stats.to_string(index=False))

## 7. DIAGRAMA ENTIDAD-RELACIÓN (DER) (Slides 31-33)

### ¿Qué es un DER?

Representación gráfica que muestra:
- Entidades (rectángulos)
- Atributos (óvalos)
- Relaciones (rombos)
- Cardinalidades (1:1, 1:N, N:M)

In [ ]:
der_diagram = """
    DIAGRAMA ENTIDAD-RELACIÓN - SISTEMA ACADÉMICO
    
    
                         ┌──────────────┐
                         │   DOCENTE    │
                         ├──────────────┤
                         │ id_docente   │
                         │ nombre       │
                         │ especialidad │
                         └──────────────┘
                               │
                            1  │  N
                               │
                         ┌──────────────┐
                         │    CURSO     │
                         ├──────────────┤
                         │ id_curso     │
                         │ nombre       │
                         │ horas        │
                         └──────────────┘
                               │
                            N  │  M
                               │
                         ┌──────────────┐       1      ┌─────────────┐
                         │  MATRÍCULA   │◄────────────→│ ESTUDIANTE  │
                         ├──────────────┤              ├─────────────┤
                         │ id_matrícula │              │ id_estudiante
                         │ fecha_inscr  │              │ nombre
                         └──────────────┘              │ correo
                                                        │ fecha_nacim
                                                        └─────────────┘
                                                               │
                                                            1  │  1
                                                               │
                                                        ┌──────────────┐
                                                        │  CONTRATO    │
                                                        ├──────────────┤
                                                        │ id_contrato  │
                                                        │ numero_contrato
                                                        │ fecha_inicio │
                                                        └──────────────┘
"""

print(der_diagram)

## 8. VALIDACIÓN CON INTEGRIDAD REFERENCIAL

In [ ]:
# Verificar que las relaciones están correctamente establecidas
print("VALIDACIÓN DE INTEGRIDAD REFERENCIAL:")
print("="*70)

# Verificar clientes
df_estudiantes_count = pd.read_sql_query('SELECT COUNT(*) as total FROM estudiante', conn)
print(f"\n✓ Estudiantes: {df_estudiantes_count.iloc[0,0]}")

# Verificar que no hay matrículas huérfanas
df_matriculas_validas = pd.read_sql_query(
    '''SELECT COUNT(*) as total FROM matricula m
       WHERE m.id_estudiante IN (SELECT id_estudiante FROM estudiante)
       AND m.id_curso IN (SELECT id_curso FROM curso)''',
    conn
)
print(f"✓ Matrículas válidas (no huérfanas): {df_matriculas_validas.iloc[0,0]}")

# Verificar cursos asignados a docentes válidos
df_cursos_validos = pd.read_sql_query(
    '''SELECT COUNT(*) as total FROM curso c
       WHERE c.id_docente IN (SELECT id_docente FROM docente)''',
    conn
)
print(f"✓ Cursos con docente válido: {df_cursos_validos.iloc[0,0]}")

print("\n✅ INTEGRIDAD REFERENCIAL VALIDADA")

## 9. BUENAS PRÁCTICAS (Slides 17, 20, 30, 33)

In [ ]:
buenas_practicas = """
    BUENAS PRÁCTICAS EN DISEÑO DE MODELOS
    
    AL DEFINIR ENTIDADES (Slide 17):
    ✓ Reunir usuarios clave para validarlas
    ✓ Evitar múltiples conceptos en una entidad
    ✓ Nombrar en singular (Cliente, no Clientes)
    
    AL DEFINIR ATRIBUTOS (Slide 20):
    ✓ Incluir solo atributos necesarios
    ✓ Definir correctamente tipos de datos
    ✓ Validar con TI si hay campos obligatorios
    
    AL ESTABLECER RELACIONES (Slide 30):
    ✓ Usar entidades intermedias claramente nombradas
    ✓ Evitar relaciones no justificadas
    ✓ Establecer restricciones de integridad referencial
    ✓ Indexar las claves foráneas
    
    AL CREAR DER (Slide 33):
    ✓ Usar nomenclatura clara
    ✓ Incluir todos los identificadores
    ✓ Confirmar cardinalidad con usuarios
    ✓ Actualizar cuando cambian requerimientos
"""

print(buenas_practicas)

## 10. ERRORES COMUNES A EVITAR

In [ ]:
errores_comunes = """
    ERRORES COMUNES EN DISEÑO DE MODELOS
    
    AL IDENTIFICAR ENTIDADES (Slide 17):
    ✗ Crear una entidad para cada formulario
    ✗ Modelar datos no persistentes
    
    AL DEFINIR ATRIBUTOS (Slide 20):
    ✗ Incluir atributos redundantes (edad Y fecha_nacimiento)
    ✗ Usar tipos de datos incorrectos (fechas como texto)
    ✗ Omitir atributos clave para futuras integraciones
    
    AL ESTABLECER RELACIONES (Slide 30):
    ✗ Duplicar atributos en lugar de crear relaciones
    ✗ No considerar la cardinalidad real del negocio
    ✗ No validar relaciones con equipos funcionales
    ✗ Asumir que todas las relaciones deben ser explícitas
    
    AL CREAR DER (Slide 33):
    ✗ Omitir entidades clave
    ✗ No indicar claves primarias o relaciones opcionales
    ✗ No actualizar cuando cambian requerimientos
    ✗ Usar notaciones inconsistentes
"""

print(errores_comunes)

## RESUMEN

✅ **Entidades:** Objetos del mundo real (Estudiante, Curso)  
✅ **Atributos:** Propiedades de las entidades (nombre, email)  
✅ **Identificadores:** Claves primarias únicas (id_estudiante)  
✅ **Relaciones:** Vínculos entre entidades (1:1, 1:N, N:M)  
✅ **DER:** Representación gráfica del modelo  

Un buen diseño de modelo de datos es la **base de una base de datos eficiente, escalable y mantenible**.